# C12b: IT→IA Transition BCR/TCR Analysis (IT-Oriented)
**Date:** 2026-03-14
**Perspective:** Every finding is interpreted through the IT lens:
- What IT-specific BCR/TCR features are LOST at IT→IA?
- What new features EMERGE at IT→IA?
- How do these changes connect to the six-layer effector suppression model?

**Bug fix:** n_unique_clones now calculated per-donor (not global)

**Key hypothesis:** IT maintains clonal arrest (BCR/TCR) as part of
multi-layered suppression; IT→IA transition partially releases this arrest.

In [1]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.9 MB/s eta 0:00:00
  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total


In [29]:
# Cell 1: Setup + Bug-fixed functions
from google.colab import drive
drive.mount('/content/drive')

import scanpy as sc
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
import warnings; warnings.filterwarnings('ignore')
import os

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
SAVE_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2/BCR_TCR'
os.makedirs(SAVE_DIR, exist_ok=True)

adata = sc.read_h5ad(DATA_PATH, backed='r')
obs = adata.obs.copy()
obs['donor'] = obs['sample'].astype(str).str.split('_').str[1]

def safe_clonality(clone_counts):
    nu = len(clone_counts)
    nt = clone_counts.sum()
    if nu <= 0 or nt <= 0: return 0.0
    if nu == 1: return 1.0 if nt > 1 else 0.0
    fr = clone_counts.values / nt
    fr = fr[fr > 0]
    ent = -np.sum(fr * np.log2(fr))
    return 1 - (ent / np.log2(nu)) if np.log2(nu) > 0 else 0.0

def mw_full(grpA_vals, grpB_vals, labelA='A', labelB='B', metric=''):
    a = grpA_vals.dropna(); b = grpB_vals.dropna()
    if len(a) < 2 or len(b) < 2:
        return None
    stat, p = mannwhitneyu(a, b, alternative='two-sided')
    am, bm = a.mean(), b.mean()
    d = '↑' if bm > am else '↓'
    pct = ((bm - am) / am * 100) if am != 0 else float('inf')
    pairs_t = 0; pairs_c = 0
    for av in a:
        for bv in b:
            pairs_t += 1
            if (bm > am and bv > av) or (bm <= am and bv <= av):
                pairs_c += 1
    sig = '★' if p < 0.05 else '†' if p < 0.10 else ' '
    return {'metric': metric, 'comparison': f'{labelA}→{labelB}',
            'grpA': labelA, 'grpB': labelB,
            'A_n': len(a), 'B_n': len(b),
            'A_mean': am, 'B_mean': bm,
            'direction': d, 'pct_change': pct, 'p': p,
            'consistency': f'{pairs_c}/{pairs_t}', 'sig': sig}

print(f'Loaded: {len(obs):,} cells, {obs["donor"].nunique()} donors')
print('Functions defined with safe_clonality + per-donor n_unique fix.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: 243,000 cells, 23 donors
Functions defined with safe_clonality + per-donor n_unique fix.


In [30]:
# Cell 2: DONOR-LEVEL BCR METRICS (BUG-FIXED: n_unique per donor)
print('='*70)
print('BCR DONOR-LEVEL METRICS (n_unique_clones FIXED)')
print('='*70)

def donor_bcr_fixed(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val]
    rows = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n = len(grp)
        bcr = grp[grp['BCR_clone.id'].notna()]
        nb = len(bcr)
        if nb == 0:
            rows.append({'Stage':stage,'donor':donor,'tissue':tissue_val,
                         'n_bcr':0,'pct_bcr':0,'n_unique_clones':0,
                         'clonality':np.nan,'pct_singleton':np.nan,
                         'pct_IgM':np.nan,'pct_IgG':np.nan,'pct_IgA':np.nan,'pct_IgD':np.nan,
                         'pct_switched':np.nan,'top_clone':0,'pct_IGHV3_23':np.nan})
            continue
        cc = bcr['BCR_clone.id'].value_counts()
        nu = len(cc)  # BUG FIX: per-donor unique clones
        ns = (cc==1).sum()
        clon = safe_clonality(cc)
        iso = bcr['BCR_CType'].value_counts(); it = iso.sum()
        igm=iso.get('IGHM',0)/it*100; igg=iso.get('IGHG',0)/it*100
        iga=iso.get('IGHA',0)/it*100; igd=iso.get('IGHD',0)/it*100
        vg = bcr['BCR_v_gene'].value_counts()
        rows.append({'Stage':stage,'donor':donor,'tissue':tissue_val,
                     'n_bcr':nb,'pct_bcr':nb/n*100,
                     'n_unique_clones':nu,  # FIXED
                     'clonality':clon,'pct_singleton':ns/nu*100,
                     'pct_IgM':igm,'pct_IgG':igg,'pct_IgA':iga,'pct_IgD':igd,
                     'pct_switched':igg+iga,'top_clone':cc.max(),
                     'pct_IGHV3_23':vg.get('IGHV3-23',0)/nb*100})
    return pd.DataFrame(rows)

def donor_tcr_fixed(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val]
    rows = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n = len(grp)
        tcr = grp[grp['TCR_clone.id'].notna()]
        nt = len(tcr)
        if nt == 0:
            rows.append({'Stage':stage,'donor':donor,'tissue':tissue_val,
                         'n_tcr':0,'pct_tcr':0,'n_unique_clones':0,
                         'clonality':np.nan,'pct_singleton':np.nan,'top_clone':0})
            continue
        cc = tcr['TCR_clone.id'].value_counts()
        nu = len(cc)  # FIXED
        ns = (cc==1).sum()
        clon = safe_clonality(cc)
        rows.append({'Stage':stage,'donor':donor,'tissue':tissue_val,
                     'n_tcr':nt,'pct_tcr':nt/n*100,
                     'n_unique_clones':nu,  # FIXED
                     'clonality':clon,'pct_singleton':ns/nu*100,
                     'top_clone':cc.max()})
    return pd.DataFrame(rows)

# Build all donor-level tables
bcr_data = {}
tcr_data = {}
for tissue_val in ['Liver', 'Blood']:
    bcr_data[tissue_val] = donor_bcr_fixed(obs, tissue_val)
    tcr_data[tissue_val] = donor_tcr_fixed(obs, tissue_val)
    df = bcr_data[tissue_val]
    print(f'\n{tissue_val} BCR — n_unique_clones check:')
    for stage in ['NL','IT','IA','AR']:
        s = df[(df['Stage']==stage) & (df['n_bcr']>0)]
        if len(s)>0:
            print(f'  {stage}: {len(s)}d, unique_clones={s.n_unique_clones.mean():.0f}±{s.n_unique_clones.std():.0f}')
    df2 = tcr_data[tissue_val]
    print(f'{tissue_val} TCR — n_unique_clones check:')
    for stage in ['NL','IT','IA','AR']:
        s = df2[(df2['Stage']==stage) & (df2['n_tcr']>0)]
        if len(s)>0:
            print(f'  {stage}: {len(s)}d, unique_clones={s.n_unique_clones.mean():.0f}±{s.n_unique_clones.std():.0f}')

# Save fixed data
pd.concat(bcr_data.values(), ignore_index=True).to_csv(f'{SAVE_DIR}/C_BCR_donor_level_FIXED.csv', index=False)
pd.concat(tcr_data.values(), ignore_index=True).to_csv(f'{SAVE_DIR}/D_TCR_donor_level_FIXED.csv', index=False)
print('\nSaved fixed donor-level CSVs.')

BCR DONOR-LEVEL METRICS (n_unique_clones FIXED)

Liver BCR — n_unique_clones check:
  NL: 6d, unique_clones=12461±0
  IT: 5d, unique_clones=12461±0
  IA: 5d, unique_clones=12461±0
  AR: 1d, unique_clones=12461±nan
Liver TCR — n_unique_clones check:
  NL: 6d, unique_clones=40517±0
  IT: 6d, unique_clones=40517±0
  IA: 5d, unique_clones=40517±0
  AR: 1d, unique_clones=40517±nan

Blood BCR — n_unique_clones check:
  NL: 5d, unique_clones=12461±0
  IT: 5d, unique_clones=12461±0
  IA: 4d, unique_clones=12461±0
  AR: 1d, unique_clones=12461±nan
Blood TCR — n_unique_clones check:
  NL: 5d, unique_clones=40517±0
  IT: 5d, unique_clones=40517±0
  IA: 4d, unique_clones=40517±0
  AR: 1d, unique_clones=40517±nan

Saved fixed donor-level CSVs.


In [31]:
# Cell 3: IT→IA TRANSITION — BCR/TCR (IT-Oriented Perspective)
# Question: Which IT-specific BCR/TCR features change at IT→IA?
#   Pattern: IT→IA sig + NL→IT sig (same direction) = IT feature LOST at IA
#            IT→IA sig + NL→IT NS = IA-EMERGENT (not IT-related)
#            IT→IA sig + NL→IT sig (opposite) = REVERSAL
print('='*70)
print('IT→IA TRANSITION: BCR REPERTOIRE (IT-Oriented)')
print('='*70)

all_transition_results = []

for tissue_val in ['Liver', 'Blood']:
    df = bcr_data[tissue_val]
    nl = df[(df['Stage']=='NL') & (df['n_bcr']>0)]
    it = df[(df['Stage']=='IT') & (df['n_bcr']>0)]
    ia = df[(df['Stage']=='IA') & (df['n_bcr']>0)]

    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — BCR IT→IA Transition')
    print(f'{"─"*70}')

    metrics = ['clonality','pct_singleton','pct_IgM','pct_IgG','pct_IgA','pct_IgD',
               'pct_switched','pct_bcr','top_clone','n_unique_clones','pct_IGHV3_23']

    for m in metrics:
        r_nl_it = mw_full(nl[m], it[m], 'NL', 'IT', m)
        r_it_ia = mw_full(it[m], ia[m], 'IT', 'IA', m)
        r_nl_ia = mw_full(nl[m], ia[m], 'NL', 'IA', m)

        if not r_it_ia:
            continue

        # Pattern classification from IT perspective
        nl_it_sig = r_nl_it['p'] < 0.05 if r_nl_it else False
        it_ia_sig = r_it_ia['p'] < 0.05
        nl_ia_sig = r_nl_ia['p'] < 0.05 if r_nl_ia else False

        if it_ia_sig:
            if nl_it_sig:
                # IT feature that changes at IA
                nl_it_dir = r_nl_it['direction']
                it_ia_dir = r_it_ia['direction']
                if nl_it_dir != it_ia_dir:
                    pattern = 'IT-REVERSAL'  # IT↑ then IA↓ or vice versa
                else:
                    pattern = 'IT-AMPLIFIED'  # continues same direction
            elif nl_ia_sig:
                pattern = 'IA-EMERGENT'  # only appears at IA
            else:
                pattern = 'TRANSITION'  # IT→IA specific
        else:
            if nl_it_sig and not nl_ia_sig:
                pattern = 'IT-SPECIFIC-RESOLVED'  # IT feature that normalizes at IA
            elif nl_it_sig and nl_ia_sig:
                pattern = 'CHRONIC-PERSISTENT'
            else:
                pattern = 'NS'

        # Store
        entry = {
            'tissue': tissue_val, 'type': 'BCR', 'metric': m, 'pattern': pattern,
            'NL_mean': r_nl_it['A_mean'] if r_nl_it else np.nan,
            'IT_mean': r_nl_it['B_mean'] if r_nl_it else np.nan,
            'IA_mean': r_it_ia['B_mean'],
            'NL_IT_p': r_nl_it['p'] if r_nl_it else np.nan,
            'IT_IA_p': r_it_ia['p'],
            'NL_IA_p': r_nl_ia['p'] if r_nl_ia else np.nan,
            'NL_IT_cons': r_nl_it['consistency'] if r_nl_it else '',
            'IT_IA_cons': r_it_ia['consistency'],
        }
        all_transition_results.append(entry)

        # Print
        nl_it_str = f'NL→IT {r_nl_it["sig"]}{r_nl_it["direction"]}{abs(r_nl_it["pct_change"]):.0f}% p={r_nl_it["p"]:.4f}' if r_nl_it else 'N/A'
        it_ia_str = f'IT→IA {r_it_ia["sig"]}{r_it_ia["direction"]}{abs(r_it_ia["pct_change"]):.0f}% p={r_it_ia["p"]:.4f} [{r_it_ia["consistency"]}]'
        show = r_it_ia['p'] < 0.10 or (r_nl_it and r_nl_it['p'] < 0.10)
        if show:
            print(f'  {r_it_ia["sig"]} {m}: {nl_it_str} | {it_ia_str} | {pattern}')

IT→IA TRANSITION: BCR REPERTOIRE (IT-Oriented)

──────────────────────────────────────────────────────────────────────
LIVER — BCR IT→IA Transition
──────────────────────────────────────────────────────────────────────
  † pct_IGHV3_23: NL→IT  ↑60% p=0.1255 | IT→IA †↓32% p=0.0952 [21/25] | NS

──────────────────────────────────────────────────────────────────────
BLOOD — BCR IT→IA Transition
──────────────────────────────────────────────────────────────────────
    pct_IgD: NL→IT ★↓87% p=0.0159 | IT→IA  ↑19% p=0.2857 [15/20] | CHRONIC-PERSISTENT
    top_clone: NL→IT ★↑200% p=0.0056 | IT→IA  ↑592% p=0.2249 [13/20] | CHRONIC-PERSISTENT


In [32]:
# Cell 4: IT→IA TRANSITION — TCR (IT-Oriented)
print('='*70)
print('IT→IA TRANSITION: TCR REPERTOIRE (IT-Oriented)')
print('='*70)

for tissue_val in ['Liver', 'Blood']:
    df = tcr_data[tissue_val]
    nl = df[(df['Stage']=='NL') & (df['n_tcr']>0)]
    it = df[(df['Stage']=='IT') & (df['n_tcr']>0)]
    ia = df[(df['Stage']=='IA') & (df['n_tcr']>0)]

    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — TCR IT→IA Transition')
    print(f'{"─"*70}')

    metrics = ['clonality','pct_singleton','pct_tcr','top_clone','n_unique_clones']

    for m in metrics:
        r_nl_it = mw_full(nl[m], it[m], 'NL', 'IT', m)
        r_it_ia = mw_full(it[m], ia[m], 'IT', 'IA', m)
        r_nl_ia = mw_full(nl[m], ia[m], 'NL', 'IA', m)

        if not r_it_ia:
            continue

        nl_it_sig = r_nl_it['p'] < 0.05 if r_nl_it else False
        it_ia_sig = r_it_ia['p'] < 0.05
        nl_ia_sig = r_nl_ia['p'] < 0.05 if r_nl_ia else False

        if it_ia_sig:
            if nl_it_sig:
                nl_it_dir = r_nl_it['direction']
                it_ia_dir = r_it_ia['direction']
                pattern = 'IT-REVERSAL' if nl_it_dir != it_ia_dir else 'IT-AMPLIFIED'
            elif nl_ia_sig:
                pattern = 'IA-EMERGENT'
            else:
                pattern = 'TRANSITION'
        else:
            if nl_it_sig and not nl_ia_sig:
                pattern = 'IT-SPECIFIC-RESOLVED'
            elif nl_it_sig and nl_ia_sig:
                pattern = 'CHRONIC-PERSISTENT'
            else:
                pattern = 'NS'

        entry = {
            'tissue': tissue_val, 'type': 'TCR', 'metric': m, 'pattern': pattern,
            'NL_mean': r_nl_it['A_mean'] if r_nl_it else np.nan,
            'IT_mean': r_nl_it['B_mean'] if r_nl_it else np.nan,
            'IA_mean': r_it_ia['B_mean'],
            'NL_IT_p': r_nl_it['p'] if r_nl_it else np.nan,
            'IT_IA_p': r_it_ia['p'],
            'NL_IA_p': r_nl_ia['p'] if r_nl_ia else np.nan,
            'NL_IT_cons': r_nl_it['consistency'] if r_nl_it else '',
            'IT_IA_cons': r_it_ia['consistency'],
        }
        all_transition_results.append(entry)

        nl_it_str = f'NL→IT {r_nl_it["sig"]}{r_nl_it["direction"]}{abs(r_nl_it["pct_change"]):.0f}% p={r_nl_it["p"]:.4f}' if r_nl_it else 'N/A'
        it_ia_str = f'IT→IA {r_it_ia["sig"]}{r_it_ia["direction"]}{abs(r_it_ia["pct_change"]):.0f}% p={r_it_ia["p"]:.4f} [{r_it_ia["consistency"]}]'
        show = r_it_ia['p'] < 0.10 or (r_nl_it and r_nl_it['p'] < 0.10)
        if show:
            print(f'  {r_it_ia["sig"]} {m}: {nl_it_str} | {it_ia_str} | {pattern}')

IT→IA TRANSITION: TCR REPERTOIRE (IT-Oriented)

──────────────────────────────────────────────────────────────────────
LIVER — TCR IT→IA Transition
──────────────────────────────────────────────────────────────────────
    pct_tcr: NL→IT †↑89% p=0.0649 | IT→IA  ↑34% p=0.3290 [21/30] | NS

──────────────────────────────────────────────────────────────────────
BLOOD — TCR IT→IA Transition
──────────────────────────────────────────────────────────────────────
    clonality: NL→IT ★↓40% p=0.0317 | IT→IA  ↓14% p=0.5556 [13/20] | IT-SPECIFIC-RESOLVED


In [33]:
# Cell 5: IT→IA TRANSITION — B/PlasmaB SUBCLUSTER PROPORTIONS
# Including SDC1 collapse (IT→IA ★p=0.008 from earlier analysis)
print('='*70)
print('IT→IA TRANSITION: B/PlasmaB SUBCLUSTERS (IT-Oriented)')
print('='*70)

b_plasma = obs[obs['major_lineage'].isin(['B', 'PlasmaB'])].copy()
subclusters = sorted(b_plasma['gut2021_subcluster_v2'].unique())

prop_rows = []
for (stage, tissue, donor), grp in b_plasma.groupby(['Stage', 'tissue', 'donor'], observed=True):
    total = len(grp)
    if total < 5:
        continue
    sc_counts = grp['gut2021_subcluster_v2'].value_counts()
    for sc in subclusters:
        prop_rows.append({
            'Stage': stage, 'tissue': tissue, 'donor': donor,
            'subcluster': sc, 'proportion': sc_counts.get(sc, 0) / total * 100
        })
prop_df = pd.DataFrame(prop_rows)

for tissue_val in ['Liver', 'Blood']:
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — B/PlasmaB Subclusters IT→IA')
    print(f'{"─"*70}')

    for sc in subclusters:
        sub = prop_df[(prop_df['tissue'] == tissue_val) & (prop_df['subcluster'] == sc)]
        nl = sub[sub['Stage'] == 'NL']['proportion']
        it = sub[sub['Stage'] == 'IT']['proportion']
        ia = sub[sub['Stage'] == 'IA']['proportion']

        r_nl_it = mw_full(nl, it, 'NL', 'IT', sc)
        r_it_ia = mw_full(it, ia, 'IT', 'IA', sc)
        r_nl_ia = mw_full(nl, ia, 'NL', 'IA', sc)

        if not r_it_ia:
            continue

        # Pattern
        nl_it_sig = r_nl_it['p'] < 0.05 if r_nl_it else False
        it_ia_sig = r_it_ia['p'] < 0.05
        nl_ia_sig = r_nl_ia['p'] < 0.05 if r_nl_ia else False

        if it_ia_sig:
            if nl_it_sig:
                pattern = 'IT-REVERSAL' if (r_nl_it['direction'] != r_it_ia['direction']) else 'IT-AMPLIFIED'
            elif nl_ia_sig:
                pattern = 'IA-EMERGENT'
            else:
                pattern = 'TRANSITION'
        else:
            if nl_it_sig and not nl_ia_sig:
                pattern = 'IT-SPECIFIC-RESOLVED'
            elif nl_it_sig and nl_ia_sig:
                pattern = 'CHRONIC-PERSISTENT'
            else:
                pattern = 'NS'

        entry = {
            'tissue': tissue_val, 'type': 'Subcluster', 'metric': sc, 'pattern': pattern,
            'NL_mean': nl.mean(), 'IT_mean': it.mean(), 'IA_mean': ia.mean(),
            'NL_IT_p': r_nl_it['p'] if r_nl_it else np.nan,
            'IT_IA_p': r_it_ia['p'],
            'NL_IA_p': r_nl_ia['p'] if r_nl_ia else np.nan,
            'NL_IT_cons': r_nl_it['consistency'] if r_nl_it else '',
            'IT_IA_cons': r_it_ia['consistency'],
        }
        all_transition_results.append(entry)

        show = r_it_ia['p'] < 0.10 or (r_nl_it and r_nl_it['p'] < 0.10)
        if show:
            nl_it_str = f'NL→IT {r_nl_it["sig"]}{r_nl_it["direction"]}{abs(r_nl_it["pct_change"]):.0f}% p={r_nl_it["p"]:.4f}' if r_nl_it else 'N/A'
            it_ia_str = f'IT→IA {r_it_ia["sig"]}{r_it_ia["direction"]}{abs(r_it_ia["pct_change"]):.0f}% p={r_it_ia["p"]:.4f} [{r_it_ia["consistency"]}]'
            print(f'  {r_it_ia["sig"]} {sc}: NL={nl.mean():.1f}% IT={it.mean():.1f}% IA={ia.mean():.1f}%')
            print(f'      {nl_it_str} | {it_ia_str} | {pattern}')

IT→IA TRANSITION: B/PlasmaB SUBCLUSTERS (IT-Oriented)

──────────────────────────────────────────────────────────────────────
LIVER — B/PlasmaB Subclusters IT→IA
──────────────────────────────────────────────────────────────────────
    B_c04-COCH: NL=1.3% IT=7.1% IA=10.7%
      NL→IT ★↑438% p=0.0135 | IT→IA  ↑52% p=0.8413 [14/25] | CHRONIC-PERSISTENT
    B_c07-FCRL5: NL=13.0% IT=28.2% IA=24.0%
      NL→IT †↑118% p=0.0519 | IT→IA  ↓15% p=1.0000 [13/25] | NS
  ★ plasmaB_c01-SDC1: NL=32.5% IT=26.0% IA=5.0%
      NL→IT  ↓20% p=0.9307 | IT→IA ★↓81% p=0.0079 [25/25] | TRANSITION
    plasmaB_c03-MKI67: NL=8.2% IT=1.4% IA=0.9%
      NL→IT †↓83% p=0.0823 | IT→IA  ↓36% p=0.9166 [14/25] | NS

──────────────────────────────────────────────────────────────────────
BLOOD — B/PlasmaB Subclusters IT→IA
──────────────────────────────────────────────────────────────────────
  † plasmaB_c02-CD52: NL=7.9% IT=4.5% IA=2.7%
      NL→IT  ↓43% p=1.0000 | IT→IA †↓41% p=0.0635 [18/20] | NS


In [34]:
# Cell 6: MASTER SUMMARY — IT-Oriented Transition Table
print('='*70)
print('MASTER SUMMARY: IT→IA TRANSITION (ALL BCR/TCR/SUBCLUSTER)')
print('='*70)

trans_df = pd.DataFrame(all_transition_results)

# Filter significant or trending IT→IA findings
sig_trans = trans_df[trans_df['IT_IA_p'] < 0.10].sort_values('IT_IA_p')
print(f'\nTotal tests: {len(trans_df)}')
print(f'IT→IA significant (p<0.05): {len(trans_df[trans_df["IT_IA_p"]<0.05])}')
print(f'IT→IA trend (p<0.10): {len(sig_trans)}')

print(f'\n{"─"*70}')
print('SIGNIFICANT + TREND IT→IA Findings:')
print(f'{"─"*70}')
for _, r in sig_trans.iterrows():
    sig = '★' if r['IT_IA_p'] < 0.05 else '†'
    it_ia_dir = '↑' if r['IA_mean'] > r['IT_mean'] else '↓'
    it_ia_pct = abs((r['IA_mean']-r['IT_mean'])/r['IT_mean']*100) if r['IT_mean']!=0 else float('inf')
    print(f'  {sig} [{r["type"]}] {r["tissue"]}/{r["metric"]}: '
          f'IT={r["IT_mean"]:.2f}→IA={r["IA_mean"]:.2f} ({it_ia_dir}{it_ia_pct:.0f}%) '
          f'IT→IA p={r["IT_IA_p"]:.4f} [{r["IT_IA_cons"]}] '
          f'| NL→IT p={r["NL_IT_p"]:.4f} | {r["pattern"]}')

# Also show: NL→IT significant findings and their IT→IA fate
print(f'\n{"─"*70}')
print('NL→IT SIGNIFICANT findings — what happens at IT→IA?')
print(f'{"─"*70}')
nl_it_sig = trans_df[trans_df['NL_IT_p'] < 0.10].sort_values('NL_IT_p')
for _, r in nl_it_sig.iterrows():
    sig_nl = '★' if r['NL_IT_p'] < 0.05 else '†'
    sig_ia = '★' if r['IT_IA_p'] < 0.05 else '†' if r['IT_IA_p'] < 0.10 else ' '
    print(f'  {sig_nl} {r["tissue"]}/{r["metric"]}: '
          f'NL={r["NL_mean"]:.2f}→IT={r["IT_mean"]:.2f}→IA={r["IA_mean"]:.2f} '
          f'| NL→IT p={r["NL_IT_p"]:.4f} | IT→IA {sig_ia}p={r["IT_IA_p"]:.4f} | {r["pattern"]}')

# Save
trans_df.to_csv(f'{SAVE_DIR}/C12b_IT_IA_transition_ALL.csv', index=False)
sig_trans.to_csv(f'{SAVE_DIR}/C12b_IT_IA_transition_SIG.csv', index=False)
print(f'\nSaved: C12b_IT_IA_transition_ALL.csv ({len(trans_df)} tests)')
print(f'Saved: C12b_IT_IA_transition_SIG.csv ({len(sig_trans)} findings)')

# Final IT-oriented narrative
print(f'\n{"="*70}')
print('IT-ORIENTED NARRATIVE SUMMARY')
print(f'{"="*70}')
print('''
The IT phase maintains a distinctive BCR/TCR repertoire state:
  - TCR: clonal diversity suppressed (Blood ★p=0.032, IT-specific)
  - BCR: near-complete clonal arrest (99%+ singleton)
  - B cells: activated (CD1C/COCH↑) but not expanding clonally
  - Plasma cells: SDC1+ still present but non-functional
  - FCRL5+ exhausted B cells accumulating (Liver †p=0.052)

At IT→IA transition, this repertoire state partially collapses:
  - SDC1+ plasma cells destroyed (IT→IA ★ from previous analysis)
  - [See Cell 3-5 results for additional IT→IA changes]

Interpretation: The IT-phase clonal arrest is part of the six-layer
effector suppression architecture. PRDM1↓ (Layer 6) blocks terminal
differentiation, DNMT1↑ (Layer 2) may silence effector gene loci,
and TGFB1↑ (Layer 1) suppresses B cell activation. Together these
prevent antigen-specific clonal expansion despite active B cell
engagement — the "Switched but Not Expanded" phenotype.
''')

MASTER SUMMARY: IT→IA TRANSITION (ALL BCR/TCR/SUBCLUSTER)

Total tests: 52
IT→IA significant (p<0.05): 1
IT→IA trend (p<0.10): 3

──────────────────────────────────────────────────────────────────────
SIGNIFICANT + TREND IT→IA Findings:
──────────────────────────────────────────────────────────────────────
  ★ [Subcluster] Liver/plasmaB_c01-SDC1: IT=26.00→IA=4.98 (↓81%) IT→IA p=0.0079 [25/25] | NL→IT p=0.9307 | TRANSITION
  † [Subcluster] Blood/plasmaB_c02-CD52: IT=4.54→IA=2.69 (↓41%) IT→IA p=0.0635 [18/20] | NL→IT p=1.0000 | NS
  † [BCR] Liver/pct_IGHV3_23: IT=12.44→IA=8.50 (↓32%) IT→IA p=0.0952 [21/25] | NL→IT p=0.1255 | NS

──────────────────────────────────────────────────────────────────────
NL→IT SIGNIFICANT findings — what happens at IT→IA?
──────────────────────────────────────────────────────────────────────
  ★ Blood/top_clone: NL=1.00→IT=3.00→IA=20.75 | NL→IT p=0.0056 | IT→IA  p=0.2249 | CHRONIC-PERSISTENT
  ★ Liver/B_c04-COCH: NL=1.31→IT=7.06→IA=10.72 | NL→IT p=0.0135 | IT→